# RAW Reflection Dataset

Reproduction of the data generation pipeline of *Removing Reflections from RAW Photos*
(Kee, Pikielny, Blackburn-Matzen, Levoy — CVPR 2025), using MIT-Adobe FiveK as the
source of RAW images.

Light adds linearly on the sensor, so a photo taken through glass is `m = t + r`:
a transmitted layer plus a reflected one. That identity only holds in a linear,
scene-referred colour space, which is why every source image is decoded to XYZ
without white balance, tone curve or clamping, and why clipped photosites are
tracked explicitly — their recorded value is a lower bound, not a measurement.

Function numbers (`Func. S1`, `Func. S3`, …) refer to the paper's supplementary
material; SDK line numbers refer to the Adobe DNG SDK 1.7.

In [ ]:
%pip install --quiet rawpy exifread requests tqdm opencv-python matplotlib numpy scikit-image

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

import cv2
import exifread
import numpy as np
import rawpy
import requests
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm

DATA_DIR = Path("fivek_data")
DNG_DIR = DATA_DIR / "dng"
OUT_DIR = DATA_DIR / "simulated"
CLIP_CACHE = DATA_DIR / "clip_levels.json"
for d in (DNG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_OUTDOOR = 500          # FiveK images to download per category
N_INDOOR = 500
N_EXAMPLES = 1000        # simulated (m, t, r, c) examples to write
MAX_SIDE = 1000          # source images are downscaled to this before pooling
PATCH = 256              # resolution of a simulated example
JPEG_QUALITY = 95

TAU = 0.1329             # srgb_decode(0.4), the exposure target of Func. S1
SAT_TOL = 2e-3           # clipped fraction above which Func. S1 takes the max branch
SAT_MARGIN = 4           # ADU of guard band below the clipping level
EXPOSURE_RANGE = (0.25, 4.0)   # accepted mean of m, as a factor of TAU (Sec. D.1)
SSIM_RANGE = (0.50, 0.95)      # accepted mean SSIM between m and t (Sec. D.1)
SSIM_STD_MIN = 0.05            # rejects reflections that spread their power evenly

SEED = 1009
rng = np.random.default_rng(SEED)
rnd = random.Random(SEED)

## 1. Source images

FiveK ships 5000 DNGs converted by Adobe DNG Converter, with `location` /
`time` / `light` / `subject` labels. Glass usually separates an indoor space from
an outdoor one, so both categories are downloaded.

Two screens are applied at download time. A DNG without `AsShotWhiteXY` nor
`AsShotNeutral` has no as-shot white point and cannot be placed in XYZ. A
non-mosaiced file (sRAW, linear DNG) has been binned and luma/chroma encoded by
the camera, which makes the sensor clipping level unrecoverable — those are
dropped rather than silently mis-masked.

In [ ]:
DNG_TAGS = {"ColorMatrix1": 0xC621, "ColorMatrix2": 0xC622,
            "CameraCalibration1": 0xC623, "CameraCalibration2": 0xC624,
            "AnalogBalance": 0xC627, "AsShotNeutral": 0xC628,
            "AsShotWhiteXY": 0xC629, "CalibrationIlluminant1": 0xC65A,
            "CalibrationIlluminant2": 0xC65B, "BlackLevel": 0xC61A,
            "WhiteLevel": 0xC61D}


def dng_tag(tags, name):
    """Read a DNG tag. exifread does not name them, so we go through the hex id."""
    tag = tags.get("Image Tag 0x%04X" % DNG_TAGS[name])
    if tag is None:
        return None
    return np.array([float(v.num) / float(v.den) if hasattr(v, "num") else float(v)
                     for v in tag.values], np.float64)


def read_tags(path):
    with open(path, "rb") as f:
        return exifread.process_file(f, details=False)


def model_of(path):
    """Camera identity. rawpy exposes no model string, so it comes from EXIF."""
    tags = read_tags(path)
    return f"{tags.get('Image Make', '?')} {tags.get('Image Model', '?')}"


def is_usable(path):
    """True if the file has an as-shot white point and a Bayer mosaic."""
    tags = read_tags(path)
    if dng_tag(tags, "AsShotWhiteXY") is None and dng_tag(tags, "AsShotNeutral") is None:
        return False
    with rawpy.imread(str(path)) as raw:
        return raw.raw_image_visible.ndim == 2

In [ ]:
META_URL = "https://huggingface.co/datasets/yuukicammy/MIT-Adobe-FiveK/raw/main/training.json"

meta_path = DATA_DIR / "training.json"
if not meta_path.exists():
    resp = requests.get(META_URL, timeout=120)
    resp.raise_for_status()
    meta_path.write_bytes(resp.content)
metadata = json.loads(meta_path.read_text())

by_loc = defaultdict(list)
for name, item in metadata.items():
    by_loc[item.get("categories", {}).get("location", "unknown")].append((name, item))

selection = []
for loc, n in [("outdoor", N_OUTDOOR), ("indoor", N_INDOOR)]:
    items = list(by_loc[loc])
    rnd.shuffle(items)
    selection += [(name, item, loc) for name, item in items[:n]]


def download_dng(name, item):
    url = item["urls"]["dng"].replace("http://", "https://")
    path = DNG_DIR / f"{name}.dng"
    if path.exists() and path.stat().st_size > 0:
        return path
    with requests.get(url, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        tmp = path.with_suffix(".part")
        with open(tmp, "wb") as f:
            for chunk in resp.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(path)
    return path


dng_files = []                                   # (path, location, categories)
for name, item, loc in tqdm(selection, desc="download"):
    try:
        path = download_dng(name, item)
        if is_usable(path):
            dng_files.append((path, loc, item.get("categories", {})))
    except Exception as exc:
        print(f"skipped {name}: {exc}")

size_mb = sum(p.stat().st_size for p, _, _ in dng_files) / 1e6
print(f"{len(dng_files)} usable DNGs out of {len(selection)} ({size_mb:.0f} MB)")

## 2. DNG colour calibration

The DNG spec gives two colour matrices, calibrated under two illuminants. The
matrix that applies to a photo depends on its white point, and the white point
is recovered by inverting the matrix — so the two are found together by a fixed
point iteration started at D50 (`Func. S3` line 4, SDK `NeutralToXY`).

Interpolation between the two calibrations is linear in `1/T` (mireds), not in
`T`, because chromaticity varies roughly linearly with reciprocal temperature.

In [ ]:
# Correlated colour temperatures of the standardised EXIF LightSource values
# (DNG spec ch. 6), used to place each calibration on the mired axis.
ILLUMINANT_TEMP = {1: 6504., 2: 4100., 3: 2856., 4: 5500., 9: 5500., 10: 6504.,
                   11: 7504., 12: 6430., 13: 5050., 14: 4150., 15: 3450.,
                   17: 2856., 18: 4874., 19: 6774., 20: 5503., 21: 6504.,
                   22: 7504., 23: 5003., 24: 3200.}


def xy_to_temp(xy):
    """CCT from McCamy's approximation (1992). It differs from the spec's
    Robertson method by a few kelvin near the Planckian locus."""
    x, y = float(xy[0]), float(xy[1])
    n = (x - 0.3320) / (0.1858 - y + 1e-12)
    return 449.0 * n ** 3 + 3525.0 * n ** 2 + 6823.3 * n + 5520.33


def xy_to_xyz(xy, Y=1.0):
    """Chromaticity to tristimulus at a given luminance."""
    x, y = float(xy[0]), float(xy[1])
    return np.array([Y * x / y, Y, Y * (1.0 - x - y) / y])


def read_calibration(path):
    """Everything find_xyz_to_camera needs, read from the DNG tags."""
    tags = read_tags(path)
    cm1, cm2 = dng_tag(tags, "ColorMatrix1"), dng_tag(tags, "ColorMatrix2")
    cc1, cc2 = dng_tag(tags, "CameraCalibration1"), dng_tag(tags, "CameraCalibration2")
    il1, il2 = dng_tag(tags, "CalibrationIlluminant1"), dng_tag(tags, "CalibrationIlluminant2")
    ab = dng_tag(tags, "AnalogBalance")
    eye = np.eye(3)
    cal = {"cm1": cm1.reshape(3, 3),
           "cm2": None if cm2 is None else cm2.reshape(3, 3),
           "cc1": eye if cc1 is None else cc1.reshape(3, 3),
           "cc2": eye if cc2 is None else cc2.reshape(3, 3),
           "ab": eye if ab is None else np.diag(ab),
           "temp1": ILLUMINANT_TEMP.get(int(il1[0]) if il1 is not None else 17, 2856.),
           "temp2": ILLUMINANT_TEMP.get(int(il2[0]) if il2 is not None else 21, 6504.)}
    return cal, tags


def find_xyz_to_camera(xy, cal):
    """Func. S7. Interpolates the two calibrations at the target white point."""
    t1, t2 = cal["temp1"], cal["temp2"]
    cm1, cm2 = cal["cm1"], cal["cm2"]
    cc1, cc2 = cal["cc1"], cal["cc2"]
    if cm2 is None:
        cm, cc = cm1, cc1
    else:
        temp = xy_to_temp(xy)
        if temp <= min(t1, t2):
            g = 1.0 if t1 <= t2 else 0.0
        elif temp >= max(t1, t2):
            g = 0.0 if t1 <= t2 else 1.0
        else:
            g = (1.0 / temp - 1.0 / t2) / (1.0 / t1 - 1.0 / t2)
        cm = g * cm1 + (1.0 - g) * cm2
        cc = g * cc1 + (1.0 - g) * cc2
    return cal["ab"] @ cc @ cm


def neutral_to_xy(neutral, cal, n_iter=30, tol=1e-9):
    """SDK NeutralToXY. The matrix depends on the white point and the white point
    depends on the matrix, so iterate from D50 until it stops moving."""
    xy = np.array([0.34567, 0.35850])
    for _ in range(n_iter):
        xyz = np.linalg.solve(find_xyz_to_camera(xy, cal), neutral)
        new = np.array([xyz[0], xyz[1]]) / max(xyz.sum(), 1e-12)
        if np.abs(new - xy).max() < tol:
            return new, True
        xy = new
    return xy, False


def camera_to_xyz(path):
    """Returns the CAM -> XYZ matrix and the as-shot white point in xy.
    A DNG carries AsShotWhiteXY or AsShotNeutral; the spec guarantees one."""
    cal, tags = read_calibration(path)
    xy = dng_tag(tags, "AsShotWhiteXY")
    if xy is not None:
        xy = np.asarray(xy[:2], np.float64)
    else:
        xy, converged = neutral_to_xy(dng_tag(tags, "AsShotNeutral")[:3], cal)
        if not converged:
            print(f"NeutralToXY did not converge: {path}")
    return np.linalg.inv(find_xyz_to_camera(xy, cal)).astype(np.float32), xy

## 3. Sensor clipping level

Above some code the photosite no longer measures light, and `m = t + r` is false
there. That code is a constant of the acquisition chain — full well, amplifier,
ADC — not a property of the photo, so it is calibrated once per camera and then
looked up.

`raw.white_level` is not that constant. It is the *linearity limit*: raw data
routinely exceeds it (Nikon D700 declares 15892 while photosites reach 16383),
and on bodies where LibRaw has no calibrated value it falls back to the ADC
range (`4095`, `16383`) which can sit far above the real plateau (Nikon D70s
plateaus at 3120 out of a declared 4095).

The plateau is found from the histogram: real clipping piles tens of thousands
of photosites onto a single code, which shot noise makes impossible otherwise.
The mask threshold is the lower of the two ceilings, because LibRaw flattens
everything above `white_level` in `postprocess` regardless.

In [ ]:
def plateau(raw, min_pixels=64, spike=20):
    """Clipping code of one file, or None if the histogram tail decays normally.

    Compares the top populated code to its immediate neighbours rather than
    taking the maximum, which a single hot pixel would defeat.
    """
    counts = np.bincount(raw.raw_image_visible.ravel())
    populated = np.flatnonzero(counts >= min_pixels)
    if populated.size == 0:
        return None
    top = int(populated[-1])
    tail = counts[max(0, top - 20):top].mean()
    return float(top) if counts[top] > spike * max(tail, 1.0) else None


def calibrate_clip(files, cache=CLIP_CACHE):
    """Per-camera clipping level in raw ADU, estimated from the files that carry
    its signature. Files without a plateau are silent, not ambiguous."""
    if cache.exists():
        return json.loads(cache.read_text())
    observed = defaultdict(list)
    for path, *_ in tqdm(files, desc="clip levels"):
        with rawpy.imread(str(path)) as raw:
            value = plateau(raw)
        if value is not None:
            observed[model_of(path)].append(value)
    # The most permissive observation: this value is only ever applied to files
    # where no plateau was found, so it must not create false positives.
    table = {model: float(max(values)) for model, values in observed.items()}
    cache.write_text(json.dumps(table, indent=1))
    return table


def clip_level(raw, model, table):
    """Lower of the sensor plateau and the ceiling LibRaw imposes."""
    return float(min(raw.white_level, table.get(model, raw.white_level)))


CLIP_TABLE = calibrate_clip(dng_files)
for model, level in sorted(CLIP_TABLE.items()):
    print(f"{model[:38]:40s} {level:8.0f}")

## 4. RAW to linear XYZ

DNG stages 1 to 4: linearisation, black level, demosaic, colour matrix. Stage 4
is where the linear domain ends — the forward matrix, hue/sat maps and tone
curves that follow are rendering, and the hue/sat map in particular is a
non-linear 3D LUT that would destroy additivity.

The decode flags matter. `adjust_maximum_thr=0` keeps LibRaw's `maximum`
independent of each image's own content, without which every file would land on
a different scale — fatal when the point is to add two files. `half_size=True`
sidesteps the demosaic algorithms, several of which are non-linear.

The output is signed and unbounded on purpose. Clamping negatives would bias
dark-region noise upward (the very asymmetry the black pedestal exists to
avoid), and any per-channel clamp would break `min(t,w) + min(r,w) != min(t+r,w)`.
The paper clamps at this stage (`Func. S3` line 6 applies `Func. S6`,
`∀c, min(c, CameraWhite)`); we do not, and compensate in `Func. S1` instead.

In [ ]:
POSTPROCESS = dict(
    gamma=(1, 1),                          # no tone curve
    no_auto_bright=True,                   # no auto exposure
    output_bps=16,
    adjust_maximum_thr=0,                  # `maximum` stays image-independent
    use_camera_wb=False,
    use_auto_wb=False,
    user_wb=[1.0, 1.0, 1.0, 1.0],          # no white balance; applied later in XYZ
    output_color=rawpy.ColorSpace.raw,     # sensor RGB; we apply the DNG matrix
    half_size=True,                        # avoids the non-linear demosaic algorithms
)


def read_exposure(path):
    """Exposure e = s * ISO / N^2 (Sec. 3.1). Dividing by it turns pixel values
    into quantities proportional to scene radiance, which is what makes two
    photos from two cameras addable."""
    tags = read_tags(path)

    def tag(name, default):
        t = tags.get(name)
        if t is None or not t.values:
            return default
        v = t.values[0]
        return float(v.num) / float(v.den) if hasattr(v, "num") else float(v)

    s = tag("EXIF ExposureTime", 1 / 60)
    n = tag("EXIF FNumber", 4.0)
    g = tag("EXIF ISOSpeedRatings", 100.0)
    return s * g / max(n, 0.7) ** 2


def align_mask(mask, shape):
    """Resample a boolean mask to `shape`, keeping "at least one photosite
    clipped": INTER_AREA averages, anything non-zero becomes True. On a Bayer
    array halved by `half_size` this reproduces a 2x2 `any` exactly."""
    if mask.shape[:2] == tuple(shape[:2]):
        return mask
    out = cv2.resize(mask.astype(np.float32), (shape[1], shape[0]),
                     interpolation=cv2.INTER_AREA)
    return out > 0


def read_raw_to_xyz(path, clip_table):
    """Returns (xyz, illum, saturated).

    xyz       linear, as-shot (the illuminant colour is preserved), signed
    illum     as-shot white point as XYZ tristimulus at Y = 1
    saturated boolean mask aligned with xyz, built on the Bayer array before
              demosaicing so that interpolation cannot dilute the plateau
    """
    model = model_of(path)
    with rawpy.imread(str(path)) as raw:
        level = clip_level(raw, model, clip_table)
        saturated = raw.raw_image_visible >= level - SAT_MARGIN
        cam = raw.postprocess(**POSTPROCESS)
    cam_to_xyz, xy = camera_to_xyz(path)
    xyz = np.einsum("ij,hwj->hwi", cam_to_xyz, cam.astype(np.float32) / 65535.0)
    return xyz, xy_to_xyz(xy).astype(np.float32), align_mask(saturated, xyz.shape[:2])

## 5. Colour transforms and display

White balance is a chromatic adaptation applied in XYZ, not a diagonal gain in
camera space: Bradford works in a cone space approximating human vision, which
is what the DNG spec and ACR do.

It is applied **once, after composition**, using the transmission's illuminant
(`Func. S2`). That is why `read_raw_to_xyz` returns as-shot XYZ: each source
keeps its own illuminant colour, so a tungsten interior reflected on a daylight
shop window stays orange in the mixture. Physically the photographer meters on
their subject, not on the reflection.

In [ ]:
D50 = np.array([0.9642, 1.0, 0.8249], np.float32)

BRADFORD = np.array([[0.8951, 0.2664, -0.1614],
                     [-0.7502, 1.7135, 0.0367],
                     [0.0389, -0.0685, 1.0296]], np.float32)

XYZ_D50_TO_SRGB = np.array([[3.1338561, -1.6168667, -0.4906146],
                            [-0.9787684, 1.9161415, 0.0334540],
                            [0.0719453, -0.2289914, 1.4052427]], np.float32)


def cat_matrix(src_white, dst_white=D50):
    """Bradford chromatic adaptation: XYZ under src_white -> XYZ under dst_white."""
    s = BRADFORD @ src_white
    d = BRADFORD @ dst_white
    return (np.linalg.inv(BRADFORD) @ np.diag(d / s) @ BRADFORD).astype(np.float32)


def apply_matrix(img, M):
    return np.einsum("ij,...j->...i", M, img)


def luminance(lin_srgb):
    return lin_srgb @ np.array([0.2126, 0.7152, 0.0722], np.float32)


def srgb_encode(x):
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)


def srgb_decode(x):
    return np.where(x <= 12.92 * 0.0031308, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)


def isp(lin_srgb, ev=0.0):
    """Linear sRGB -> 8-bit image. The only place clamping is legitimate."""
    x = np.clip(lin_srgb, 0, None) * 2.0 ** ev
    return (srgb_encode(np.clip(x, 0, 1)) * 255 + 0.5).astype(np.uint8)


def auto_expose(lin, target=0.18):
    """Puts the geometric mean of the luminance on middle grey. Preview only."""
    key = float(np.exp(np.log(np.clip(luminance(lin), 1e-6, None)).mean()))
    return lin * (target / max(key, 1e-8))


def xyz_to_display(xyz, illum, white_balance=True):
    M = XYZ_D50_TO_SRGB @ cat_matrix(illum) if white_balance else XYZ_D50_TO_SRGB
    return np.clip(auto_expose(np.clip(apply_matrix(xyz, M), 0, None)), 0, 1)


def show_row(images, titles, ev=0.0, size=3.2):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(isp(img, ev))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Image pool

In [ ]:
def load_pool(files, clip_table):
    pool = []
    for path, loc, cats in tqdm(files, desc="RAW -> XYZ"):
        xyz, illum, saturated = read_raw_to_xyz(path, clip_table)
        h, w = xyz.shape[:2]
        scale = MAX_SIDE / max(h, w)
        if scale < 1:
            xyz = cv2.resize(xyz, (round(w * scale), round(h * scale)),
                             interpolation=cv2.INTER_AREA)
        pool.append({"name": path.stem,
                     "xyz": xyz,
                     "illum": illum,
                     "saturation": align_mask(saturated, xyz.shape[:2]),
                     "e": read_exposure(path),
                     "loc": loc,
                     "cats": cats})
    return pool


pool = load_pool(dng_files, CLIP_TABLE)
clipped = np.mean([p["saturation"].any() for p in pool])
print(f"{len(pool)} images | exposure e from {min(p['e'] for p in pool):.2e} "
      f"to {max(p['e'] for p in pool):.2e} | {clipped:.0%} contain clipped pixels")

## 7. Geometric synthesis

A glass pane reflects a fraction of the light that depends on the incidence
angle, which varies across the frame — hence per-pixel Fresnel rather than a
scalar. Two interfaces bounce light between them, and summing the geometric
series gives `2R/(1+R)`.

Three effects follow (Sec. B): the reflected scene is behind the focus plane so
it is defocused by a disk kernel (bokeh, not gaussian); the second interface
produces a laterally shifted ghost whose displacement is zero at normal
incidence and grows with the angle; and the whole reflection is mirrored,
because the glass flips it left/right.

In [ ]:
GLASS_IOR = 1.52         # ordinary window glass
SENSOR_W = 0.024         # sensor width in metres (full frame short side)


def fresnel_single(cos_i, n=GLASS_IOR):
    """Unpolarised Fresnel reflectance of one air-glass interface."""
    cos_i = np.clip(cos_i, 1e-6, 1.0)
    sin_t = np.sqrt(np.clip(1 - cos_i ** 2, 0, 1)) / n
    cos_t = np.sqrt(np.clip(1 - sin_t ** 2, 0, 1))
    rs = ((cos_i - n * cos_t) / (cos_i + n * cos_t)) ** 2
    rp = ((n * cos_i - cos_t) / (n * cos_i + cos_t)) ** 2
    return 0.5 * (rs + rp)


def fresnel_pane(cos_i, n=GLASS_IOR):
    """Two interfaces with internal bounces: the geometric series sums to 2R/(1+R)."""
    R = fresnel_single(cos_i, n)
    return 2 * R / (1 + R)


def sample_scene(rng, size, params=None):
    """Camera, glass plane and distances, and the per-pixel quantities they imply.
    `params` overrides individual draws, which is how the tests pin a scene."""
    p = params or {}

    def draw(key, lo, hi):
        return float(p[key]) if p.get(key) is not None else float(rng.uniform(lo, hi))

    fov = draw("fov", 40, 90)
    f_px = (size / 2) / np.tan(np.radians(fov) / 2)

    # unit ray direction per pixel
    ys, xs = np.mgrid[0:size, 0:size].astype(np.float32) - (size - 1) / 2
    d = np.stack([xs, ys, np.full_like(xs, f_px)], -1)
    d /= np.linalg.norm(d, axis=-1, keepdims=True)

    # glass normal, tilted by theta0 and rotated by phi (spherical coordinates)
    theta0 = np.radians(draw("theta0_deg", 0, 55))
    phi = draw("phi", 0, 2 * np.pi)
    nrm = np.array([np.sin(theta0) * np.cos(phi),
                    np.sin(theta0) * np.sin(phi),
                    np.cos(theta0)], np.float32)
    cos_i = np.abs(d @ nrm)

    # defocus: focus is on the transmitted subject, the reflected scene sits
    # behind the glass by mirror symmetry, so it falls outside the focus plane
    f_m = f_px / size * SENSOR_W
    N = draw("aperture", 1.8, 8.0)
    d_focus = draw("d_focus", 0.5, 8.0)
    d_glass = draw("d_glass", 0.3, max(0.4, min(3.0, d_focus)))
    d_refl = d_glass + draw("d_refl", 1.0, 50.0)
    coc = f_m ** 2 / (N * max(d_focus - f_m, 1e-3)) * abs(d_refl - d_focus) / d_refl
    defocus_px = coc / SENSOR_W * size

    # ghost: the reflection off the second face exits laterally shifted, by an
    # amount that is zero at normal incidence and grows with the angle
    h_glass = draw("h_glass", 0.003, 0.015)
    sin_i = np.sqrt(np.clip(1 - cos_i ** 2, 0, None))
    sin_t = sin_i / GLASS_IOR
    tan_t = sin_t / np.sqrt(1 - sin_t ** 2)
    ghost_px = f_px * 2 * h_glass * tan_t * cos_i / d_glass

    # shift direction per pixel: the ray projected onto the glass plane
    tang = d - (d @ nrm)[..., None] * nrm
    tang /= np.maximum(np.linalg.norm(tang, axis=-1, keepdims=True), 1e-8)
    ghost_dir = tang[..., :2].astype(np.float32)
    ghost_dir /= np.maximum(np.linalg.norm(ghost_dir, axis=-1, keepdims=True), 1e-8)

    return {"f_px": f_px,
            "R_map": fresnel_pane(cos_i)[..., None].astype(np.float32),
            "R1_map": fresnel_single(cos_i).astype(np.float32),
            "defocus_px": float(defocus_px),
            "ghost_px": ghost_px.astype(np.float32),
            "ghost_dir": ghost_dir}

In [ ]:
def disk_kernel(radius):
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    k = ((x ** 2 + y ** 2) <= radius ** 2).astype(np.float32)
    return k / k.sum()


def defocus(img, radius):
    """Disk kernel, not gaussian: an out-of-focus point images as a bokeh disk."""
    return img if radius < 1 else cv2.filter2D(img, -1, disk_kernel(int(radius)))


def double_reflection(img, scene):
    """Ghost from the second glass face, displaced by a per-pixel amount."""
    shift = scene["ghost_px"]
    if float(shift.max()) < 0.15:
        return img
    h, w = img.shape[:2]
    ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
    dxy = scene["ghost_dir"]
    ghost = cv2.remap(img,
                      (xs - dxy[..., 0] * shift).astype(np.float32),
                      (ys - dxy[..., 1] * shift).astype(np.float32),
                      cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    a = ((1.0 - scene["R1_map"]) ** 2)[..., None]      # transmitted twice
    return (img + a * ghost) / (1.0 + a)


def random_crop(img, sat, size, rng):
    """Returns the resized crop and the exact clipped fraction inside it.

    The fraction is read on the full-resolution mask before any resampling, so
    it is exact; resizing the mask would dilute isolated clipped pixels.
    """
    h, w = img.shape[:2]
    c = int(min(h, w) * rng.uniform(0.4, 1.0))
    y = int(rng.integers(0, h - c + 1))
    x = int(rng.integers(0, w - c + 1))
    interp = cv2.INTER_AREA if c >= size else cv2.INTER_LINEAR
    crop = cv2.resize(img[y:y + c, x:x + c], (size, size), interpolation=interp)
    return crop, float(sat[y:y + c, x:x + c].mean())


def split_reflection_context(img, sat, rng):
    """Disjoint halves (Sec. 3.3): one becomes the reflection, the other the
    contextual photo. Image and mask follow the same draws."""
    h, w = img.shape[:2]
    if rng.random() < 0.5:
        halves = (img[:, :w // 2], img[:, w // 2:])
        masks = (sat[:, :w // 2], sat[:, w // 2:])
    else:
        halves = (img[:h // 2], img[h // 2:])
        masks = (sat[:h // 2], sat[h // 2:])
    i = int(rng.random() < 0.5)
    return halves[i], halves[1 - i], masks[i], masks[1 - i]

## 8. Photometric synthesis

`Func. 1` in scene-referred radiance: divide each source by its exposure, apply
the glass, add, then re-expose the sum with `Func. S1`.

`Func. S1` exposes the mean of `m` to sRGB 0.4, unless a component contains
clipped pixels — a clipped value is a lower bound, so scaling it up would
amplify a number known to be wrong, and the exposure is capped instead.

Two documented deviations. `Func. S1` line 10 uses `max(t)`; we use the 99.9th
percentile, because our sources are not bounded by the `CameraWhite` clamp of
`Func. S6` and a single blown streetlight would otherwise set the exposure of
the whole patch. And the saturation test uses a fraction rather than "any
pixel", since with half the dataset containing clipped pixels the strict test
would send nearly every mixture down the capped branch.

In [ ]:
def compute_exposure(m, t, r, c, frac_t, frac_r, M):
    """Func. S1. Returns (m, t, r, c) in linear sRGB, or None if the mixture is
    badly exposed (Sec. D.1, first criterion)."""
    m_s, t_s, r_s, c_s = (apply_matrix(x, M) for x in (m, t, r, c))

    if frac_t < SAT_TOL and frac_r < SAT_TOL:
        e = TAU / max(float(m_s.mean()), 1e-8)
    else:
        hi_t = float(np.percentile(t_s, 99.9))
        hi_r = float(np.percentile(r_s, 99.9))
        e = 1.0 / max(min(hi_t, hi_r), 1e-8)

    lo, hi = EXPOSURE_RANGE
    if not lo * TAU < float(luminance(m_s * e).mean()) < hi * TAU:
        return None

    e_c = TAU / max(float(c_s.mean()), 1e-8)
    return m_s * e, t_s * e, r_s * e, c_s * e_c


def simulate_example(src_t, src_r, rng, patch=PATCH, params=None):
    """Func. 1. Returns dict(m, t, r, c) in linear sRGB, or None if culled."""
    r_src, c_src, sat_r, sat_c = split_reflection_context(
        src_r["xyz"], src_r["saturation"], rng)

    t, frac_t = random_crop(src_t["xyz"], src_t["saturation"], patch, rng)
    r, frac_r = random_crop(r_src, sat_r, patch, rng)
    c, _ = random_crop(c_src, sat_c, patch, rng)

    # scene-referred radiance: undo each camera's exposure (Sec. 3.1)
    t, r, c = t / src_t["e"], r / src_r["e"], c / src_r["e"]

    scene = sample_scene(rng, patch, params)
    if rng.random() < 0.5:
        r = r[:, ::-1].copy()                # the glass mirrors the reflection
    r = defocus(r, int(round(min(scene["defocus_px"], 12))))
    r = double_reflection(r, scene)
    r = r * scene["R_map"]                   # per-pixel Fresnel
    t = t * (1.0 - scene["R_map"])           # the glass attenuates t as well

    m = t + r                                # light adds on the sensor

    exposed = compute_exposure(m, t, r, c, frac_t, frac_r,
                               XYZ_D50_TO_SRGB @ cat_matrix(src_t["illum"]))
    if exposed is None:
        return None
    m, t, r, c = exposed
    # a real sensor would clip the composite; t and r stay unclipped so they
    # remain valid targets
    return {"m": np.clip(m, 0, 1), "t": t, "r": r, "c": c}

## 9. Pair sampling and mixture search

Reflectance is only 4 to 15 %, so a reflection is visible only when the
reflected scene is much brighter than the transmitted one. An outdoor
transmission with an indoor reflection is physically almost invisible; the
reverse is the shop-window case. The paper's pair set is therefore
`D = (O x I) u (I x O) u (I x I) - P`, with outdoor-outdoor omitted (Sec. D.2).

Even so most mixtures fail, and the paper searches rather than fixes: `Sec. D.1`
keeps a mixture if its mean is in the dataset's normal range, and if the SSIM
between `m` and `t` sits in a useful band — too high and the reflection is
imperceptible, too low and the mixture is no longer interpretable. The standard
deviation of the SSIM image removes reflections that spread their power evenly.

The thresholds are not given in the paper. Plot the distributions over a few
thousand draws and set the bounds at the troughs.

In [ ]:
def pair_score(src_t, src_r):
    """Realism weight for a (transmission, reflection) pair."""
    if src_t["loc"] == "outdoor" and src_r["loc"] == "outdoor":
        return 0.0                                   # omitted set (Sec. D.2)

    cats_t, cats_r = src_t["cats"], src_r["cats"]
    s = 4.0 if src_t["loc"] != src_r["loc"] else 0.3

    # daylight outdoors reflected on a darker interior is the strong case
    if src_r["loc"] == "outdoor":
        s *= {"day": 2.0, "dusk": 1.5, "night": 0.7}.get(cats_r.get("time"), 1.0)
    if src_r["loc"] == "indoor":
        if cats_r.get("light") == "artificial":
            s *= 1.5
        s *= {"night": 2.0, "dusk": 2.0, "day": 0.7}.get(cats_t.get("time"), 1.0)

    # different illuminants give the colour mixing that makes a mixture informative
    it = src_t["illum"] / src_t["illum"].sum()
    ir = src_r["illum"] / src_r["illum"].sum()
    s *= 1.0 + 4.0 * min(float(np.abs(it - ir).sum()), 0.25)

    # radiance ratio after exposure normalisation: the reflection must carry
    # enough energy to survive Fresnel attenuation
    ratio = (src_r["xyz"][..., 1].mean() / src_r["e"]) / \
            (src_t["xyz"][..., 1].mean() / src_t["e"] + 1e-12)
    s *= float(np.clip(ratio, 0.05, 20.0)) ** 0.25
    return s


ALL_PAIRS = [(i, j) for i in range(len(pool)) for j in range(len(pool)) if i != j]
PAIR_WEIGHTS = [pair_score(pool[i], pool[j]) for i, j in ALL_PAIRS]


def sample_realistic_pair(rnd):
    return rnd.choices(ALL_PAIRS, weights=PAIR_WEIGHTS, k=1)[0]


for w, (i, j) in sorted(zip(PAIR_WEIGHTS, ALL_PAIRS))[-10:]:
    print(f"score {w:6.2f}  t = {pool[i]['name']} ({pool[i]['loc']})"
          f"  <- r = {pool[j]['name']} ({pool[j]['loc']})")

In [ ]:
def mixture_score(m, t):
    """Sec. D.1. Block-wise SSIM between m and t, averaged across channels with
    weights equal to their mean value, which handles strongly coloured images."""
    w = np.array([m[..., k].mean() for k in range(3)], np.float64)
    w /= max(w.sum(), 1e-8)
    S = sum(w[k] * structural_similarity(m[..., k], t[..., k],
                                         data_range=1.0, full=True)[1]
            for k in range(3))
    return float(S.mean()), float(S.std())


def is_useful(example):
    mean_ssim, std_ssim = mixture_score(example["m"], example["t"])
    lo, hi = SSIM_RANGE
    if not lo < mean_ssim < hi:
        return False
    if std_ssim < SSIM_STD_MIN:                      # reflection without structure
        return False
    return (example["m"].max(axis=-1) > 0.99).mean() <= 0.08

## 10. Dataset generation

In [ ]:
def save_jpeg(path, lin_srgb):
    cv2.imwrite(str(path), isp(lin_srgb)[..., ::-1],
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])


n_saved = n_tried = 0
progress = tqdm(total=N_EXAMPLES, desc="simulation")
while n_saved < N_EXAMPLES and n_tried < 200 * N_EXAMPLES:
    n_tried += 1
    i, j = sample_realistic_pair(rnd)
    example = simulate_example(pool[i], pool[j], rng)
    if example is None or not is_useful(example):
        continue
    stem = f"ex_{n_saved:04d}"
    for key in "mtrc":
        save_jpeg(OUT_DIR / f"{stem}_{key}.jpg", example[key])
    n_saved += 1
    progress.update(1)
progress.close()
print(f"{n_saved} examples from {n_tried} attempts "
      f"({n_saved / max(n_tried, 1):.0%} accepted) -> {OUT_DIR}")

In [ ]:
def show_examples(n=3, root=None, start=0):
    import matplotlib.pyplot as plt
    root = Path(root) if root else OUT_DIR
    stems = sorted({p.name[:-6] for p in root.glob("ex_*_m.jpg")})[start:start + n]
    if not stems:
        print(f"no examples in {root}")
        return
    titles = ["mixture m", "transmission t", "reflection r", "context c"]
    fig, axes = plt.subplots(len(stems), 4, figsize=(13, 3.4 * len(stems)),
                             squeeze=False)
    for row, stem in zip(axes, stems):
        for ax, key, title in zip(row, "mtrc", titles):
            ax.imshow(cv2.imread(str(root / f"{stem}_{key}.jpg"))[..., ::-1])
            ax.set_title(f"{stem} - {title}", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


show_examples(3)

## 11. PyTorch dataset

Additivity holds in the linear domain where the mixture was built, not in the
tone-mapped sRGB written to disk — the gamma is non-linear by construction. The
check below is expected to fail, and its failing is the reminder that the
physics lives upstream of the JPEG.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


class FiveKReflections(Dataset):
    """(mixture, context) -> (transmission, reflection), in sRGB [0, 1]."""

    def __init__(self, root, patch=224, train=True):
        self.root = Path(root)
        self.stems = sorted(p.name[:-6] for p in self.root.glob("ex_*_m.jpg"))
        self.patch = patch
        self.train = train

    def _load(self, stem, key):
        img = cv2.imread(str(self.root / f"{stem}_{key}.jpg"))
        return img[..., ::-1].astype(np.float32) / 255.0

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        m, t, r, c = (self._load(stem, key) for key in "mtrc")
        if self.train:
            p = self.patch
            y = np.random.randint(0, m.shape[0] - p + 1)
            x = np.random.randint(0, m.shape[1] - p + 1)
            m, t, r = (a[y:y + p, x:x + p] for a in (m, t, r))
            c = cv2.resize(c, (p, p), interpolation=cv2.INTER_AREA)
            if np.random.rand() < 0.5:
                m, t, r, c = (a[:, ::-1] for a in (m, t, r, c))

        def to_tensor(a):
            return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1)

        return {"mixture": to_tensor(m), "context": to_tensor(c),
                "transmission": to_tensor(t), "reflection": to_tensor(r)}


ds = FiveKReflections(OUT_DIR)
batch = next(iter(DataLoader(ds, batch_size=min(4, len(ds)), shuffle=True)))
print({k: tuple(v.shape) for k, v in batch.items()})

m, t, r = (ds._load(ds.stems[0], key) for key in "mtr")
print(f"|m - clip(t+r)| in sRGB: {np.abs(np.clip(t + r, 0, 1) - m).mean():.3f} "
      f"(non-zero, as expected)")